# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
# The paper reports 71% holdout accuracy predicting page growth, with content age and days-since-update as the top signals. The methodology question I'd ask: where does the "growth" label come from, and was the holdout split grouped by client? The paper defines trend direction from a 30-day-vs-previous-30-day comparison, and if the 80/20 holdout split was random rather than grouped by client, the model could be partly learning client-specific patterns rather than a generalizable growth signal, similar to the leakage risk I had to catch in my own Week-5 model. This isn't a criticism of the finding itself, just a question about whether the reported accuracy would hold up under a client-grouped split.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
from sklearn.model_selection import train_test_split

# --- "Before": naive random split, no grouping by client ---
train_random, test_random = train_test_split(df, test_size=0.25, random_state=42)

X_train_r = train_random[feature_cols].fillna(0)
y_train_r = train_random["is_declining"]
X_test_r = test_random[feature_cols].fillna(0)
y_test_r = test_random["is_declining"]

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

model_random = LogisticRegression(random_state=42, max_iter=1000)
model_random.fit(X_train_r_scaled, y_train_r)
random_scores = model_random.predict_proba(X_test_r_scaled)[:, 1]

# Check for client overlap between train and test (the honesty problem this split has)
overlap = len(set(train_random["client_id"]) & set(test_random["client_id"]))
print(f"Clients appearing in BOTH train and test (random split): {overlap} "
      f"out of {df['client_id'].nunique()} total clients")

print("\nRandom split vs grouped split, same model, same features:")
for k in [10, 20, 50]:
    p_random = precision_at_k(random_scores, y_test_r, k)
    p_grouped = precision_at_k(model_scores, y_test, k)  # from the grouped split, Week 5
    print(f"k={k}  RANDOM split: {p_random:.3f}  |  GROUPED split (honest): {p_grouped:.3f}")

print(f"\nBase rate (random split test set): {y_test_r.mean():.3f}")
print(f"Base rate (grouped split test set): {y_test.mean():.3f}")

Clients appearing in BOTH train and test (random split): 32 out of 32 total clients

Random split vs grouped split, same model, same features:
k=10  RANDOM split: 0.700  |  GROUPED split (honest): 0.800
k=20  RANDOM split: 0.700  |  GROUPED split (honest): 0.850
k=50  RANDOM split: 0.800  |  GROUPED split (honest): 0.760

Base rate (random split test set): 0.546
Base rate (grouped split test set): 0.517


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# --- Setup: reload data (skip the git clone lines if repo already exists in this session) ---
!git clone https://github.com/Nextchingiz/Flyrank-AI-Internship.git
import os
os.chdir("Flyrank-AI-Internship")

!mkdir -p data/raw
!wget -O data/raw/content_refresh_anonymized.csv https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# --- Rebuild Week-5's grouped split ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# --- Rebuild Week-5's FINAL (corrected, non-leaky) feature set and model ---
feature_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

X_train = train_df[feature_cols].fillna(0)
y_train = train_df["is_declining"]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df["is_declining"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)
model_scores = model.predict_proba(X_test_scaled)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# ============================================================
# CELL 3 — Leakage audit (the actual deliverable for this section)
# ============================================================
final_feature_cols = feature_cols

banned_label_derived = ["trend_pct", "trend_direction", "is_declining_label", "is_declining"]
banned_windows = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                  "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
banned_ids = ["content_id", "client_id"]

print("Label-derived leaks present:", [c for c in final_feature_cols if c in banned_label_derived])
print("Overlapping-window leaks present:", [c for c in final_feature_cols if c in banned_windows])
print("ID columns present as features:", [c for c in final_feature_cols if c in banned_ids])

# The "confession" test from the skill: train WITH the suspect columns, then WITHOUT
leaky_feature_cols = final_feature_cols + banned_windows

X_train_leaky = train_df[leaky_feature_cols].fillna(0)
X_test_leaky = test_df[leaky_feature_cols].fillna(0)

scaler_leaky = StandardScaler()
X_train_leaky_scaled = scaler_leaky.fit_transform(X_train_leaky)
X_test_leaky_scaled = scaler_leaky.transform(X_test_leaky)

model_leaky = LogisticRegression(random_state=42, max_iter=1000)
model_leaky.fit(X_train_leaky_scaled, y_train)
leaky_scores = model_leaky.predict_proba(X_test_leaky_scaled)[:, 1]

print("\nLeakage collapse test (the 'confession'):")
for k in [10, 20, 50]:
    with_suspects = precision_at_k(leaky_scores, y_test, k)
    without_suspects = precision_at_k(model_scores, y_test, k)
    print(f"k={k}  WITH suspects: {with_suspects:.3f}  |  WITHOUT (final model): {without_suspects:.3f}")

print(f"\nBase rate: {y_test.mean():.3f}")

# Running the attack checklist against my final Week-5 feature set: no label-derived columns (trend_pct, trend_direction), no ID columns (content_id, client_id), and no overlapping-window columns (_last_30d/_prev_30d pairs) remain in the final model's inputs.
# To confirm this wasn't just caution for its own sake, I re-ran the "confession" test: training the same model with the suspect _last_30d/_prev_30d columns added back in produces a perfect precision@K of 1.000 at k=10, 20, and 50. That's the exact symptom the skill warns about, since these windows are what trend_direction was likely computed from in the first place, so their presence let the model reconstruct the label almost exactly rather than genuinely predict it. Removing them drops precision@K to 0.80, 0.85, and 0.76 respectively, against a base rate of 0.517, still comfortably beating the baseline, but now an honest number rather than a leaked one.

Cloning into 'Flyrank-AI-Internship'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 46 (delta 14), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 51.47 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (14/14), done.
--2026-08-03 10:34:57--  https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘data/raw/content_refresh_anonymized.csv’

data/raw/content_re 100%[===================>]   6.42M  --.-KB/s    in 0.05s   

2026-08-03 10:34:57 (131 MB/s) - ‘data/raw/content_refresh_anony

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
# On this measured test split, the model's precision@K (0.80 at k=10, 0.85 at k=20, 0.76 at k=50) is higher than the rule-based baseline's (0.1, 0.1, 0.2) on the same data and metric. This is a decision-support result: it shows the model ranks better on this particular holdout, not a guarantee of future performance or a claim about causation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.